# 14 · Reading a plate by redrawing it: SPD-Conv reader

The YOLO26s reader is kept as it is. After it:

1. **Learned plate layout:** the reader's guesses are turned into the 5 best readings, scored by the reader and by how
   common their layout (digits, districts, letters, words) is in the training labels.
2. **Redrawing:** each reading is drawn as a clean plate, fitted to the photo (corners, brightness, blur), and the
   reading whose drawing fits best is chosen.

**Inputs:** the `bangla-plate-reading` dataset and the output of notebook 12 (SPD-Conv), the best reader on plate accuracy. Attach only that one `best.pt`.

In [ ]:
%pip install -q ultralytics==8.4.46

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from ultralytics import YOLO

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

DATA = next(p for p in INPUT.rglob("bangla-plate-reading") if p.is_dir())
WEIGHTS = list(INPUT.rglob("best.pt"))
assert len(WEIGHTS) == 1, "attach only the reader-spd output (one best.pt)"
print(DATA)
print(WEIGHTS)

## Custom layers

The readers from notebooks 10, 12 and 13 contain custom layers; this file lets any of them be loaded.

In [ ]:
%%writefile /kaggle/working/plate_layers.py
import json
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.ops import deform_conv2d
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules import Conv
from ultralytics.nn.modules.block import Bottleneck
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK
import ultralytics.utils.torch_utils as torch_utils

KINDS = ["digit", "district", "letter", "word"]


# ---------------------------------------------------------------- deformable convolution (notebook 10)

class DConv(nn.Module):
    """Deformable 3x3 convolution (DCNv2) made from a pretrained Conv.

    The offsets start at zero and the mask at one, so before training it gives exactly the same output
    as the Conv it replaces; training then learns where each kernel tap should look.
    """

    def __init__(self, conv):
        super().__init__()
        c = conv.conv
        self.weight = nn.Parameter(c.weight.detach().clone())
        self.stride, self.padding, self.dilation = c.stride, c.padding, c.dilation
        taps = c.kernel_size[0] * c.kernel_size[1]
        self.offset = nn.Conv2d(c.in_channels, 3 * taps, c.kernel_size, c.stride, c.padding)
        nn.init.zeros_(self.offset.weight)
        nn.init.zeros_(self.offset.bias)
        self.bn = conv.bn
        self.act = conv.act

    def forward(self, x):
        dy_dx_1, dy_dx_2, mask = self.offset(x).chunk(3, dim=1)
        offset = torch.cat([dy_dx_1, dy_dx_2], dim=1)
        y = deform_conv2d(x, offset, self.weight, None, self.stride, self.padding, self.dilation, 2 * torch.sigmoid(mask))
        return self.act(self.bn(y))


# Ultralytics counts FLOPs with thop on an uninitialised tensor and the deformable op crashes on it.
# The count is only for the printed summary, so it is skipped for models with deformable convs.
_get_flops = torch_utils.get_flops


def _get_flops_safe(model, imgsz=640):
    if any(isinstance(m, DConv) for m in model.modules()):
        return 0.0
    return _get_flops(model, imgsz)


torch_utils.get_flops = _get_flops_safe


def add_dcn(model, layers=(6, 8)):
    """Swap the 3x3 conv of every bottleneck in backbone layers 6 and 8 (P4 and P5) for a DConv."""
    swapped = 0
    for i in layers:
        for m in model.model[i].modules():
            if isinstance(m, Bottleneck) and isinstance(m.cv2, Conv) and m.cv2.conv.kernel_size == (3, 3) \
                    and m.cv2.conv.stride == (1, 1) and m.cv2.conv.groups == 1:
                m.cv2 = DConv(m.cv2)
                swapped += 1
    return swapped


class DCNTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        print("deformable convs:", add_dcn(model))
        return model


# ---------------------------------------------------------------- SPD-Conv (notebook 12)

class SPDConv(nn.Module):
    """Space-to-depth then a stride-1 conv: halves the size without dropping any pixel (SPD-Conv)."""

    def __init__(self, c1, c2):
        super().__init__()
        self.conv = Conv(4 * c1, c2, 3, 1)

    def forward(self, x):
        x = torch.cat([x[..., ::2, ::2], x[..., 1::2, ::2], x[..., ::2, 1::2], x[..., 1::2, 1::2]], dim=1)
        return self.conv(x)


def add_spd(model, layers=(1, 3, 5, 7)):
    """Replace the stride-2 Conv layers of the backbone (the downsampling to P2, P3, P4 and P5) with SPDConv."""
    for i in layers:
        old = model.model[i]
        new = SPDConv(old.conv.in_channels, old.conv.out_channels)
        new.f, new.i, new.type = old.f, old.i, "SPDConv"
        new.np = sum(p.numel() for p in new.parameters())
        model.model[i] = new
    return len(layers)


class SPDTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        print("SPD layers:", add_spd(model))
        return model


# ---------------------------------------------------------------- layout head (notebook 13)

class Tap(nn.Module):
    """Wraps one backbone layer and keeps its last output, so the layout head can read it."""

    def __init__(self, m):
        super().__init__()
        self.m = m
        self.f, self.i, self.type, self.np = m.f, m.i, m.type, m.np
        self.out = None

    def forward(self, x):
        self.out = self.m(x)
        return self.out

    def __getstate__(self):
        state = self.__dict__.copy()
        state["out"] = None   # never save a feature map with the model
        return state


class LayoutHead(nn.Module):
    """Predicts how many digits, districts, letters and words the plate has, from pooled backbone features."""

    def __init__(self, c, caps, hidden=256):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(2 * c, hidden), nn.SiLU())
        self.heads = nn.ModuleList(nn.Linear(hidden, cap + 1) for cap in caps)

    def forward(self, x):
        x = torch.cat([F.adaptive_avg_pool2d(x, 1), F.adaptive_max_pool2d(x, 1)], 1).flatten(1)
        h = self.fc(x)
        return [head(h) for head in self.heads]


def plate_layout_info(data_dir):
    """Kind of every class (0 digit, 1 district, 2 letter, 3 word) and the largest count of each kind in training."""
    data_dir = Path(data_dir)
    classes = json.loads((data_dir / "classes.json").read_text(encoding="utf-8"))
    kinds = [KINDS.index(c["kind"]) for c in classes]
    caps = [0, 0, 0, 0]
    for label in (data_dir / "labels" / "train").glob("*.txt"):
        counts = [0, 0, 0, 0]
        for line in label.read_text().splitlines():
            if line.strip():
                counts[kinds[int(line.split()[0])]] += 1
        caps = [max(a, b) for a, b in zip(caps, counts)]
    return kinds, caps


class LayoutModel(DetectionModel):
    """YOLO detector with an extra head that predicts the plate layout, trained together with detection."""

    def add_layout(self, kinds, caps, layer=10, weight=1.0):
        self.model[layer] = Tap(self.model[layer])
        self.layout_layer = layer
        self.layout_caps = list(caps)
        self.layout_weight = weight
        self.register_buffer("kind_index", torch.tensor(kinds, dtype=torch.long))
        with torch.no_grad():
            p = next(self.parameters())
            s = int(self.stride.max())
            self.model.eval()
            self.forward(torch.zeros(1, 3, 2 * s, 2 * s, device=p.device, dtype=p.dtype))
            self.model.train()
        c = self.model[layer].out.shape[1]
        self.model[layer].out = None
        self.layout_head = LayoutHead(c, caps)

    def layout_logits(self):
        return self.layout_head(self.model[self.layout_layer].out)

    def layout_targets(self, batch, n):
        counts = torch.zeros(n, 4, dtype=torch.long, device=self.kind_index.device)
        if len(batch["cls"]):
            img = batch["batch_idx"].long().to(counts.device)
            kind = self.kind_index[batch["cls"].view(-1).long().to(counts.device)]
            counts.index_put_((img, kind), torch.ones_like(img), accumulate=True)
        caps = torch.tensor(self.layout_caps, device=counts.device)
        return torch.minimum(counts, caps)

    def loss(self, batch, preds=None):
        if preds is None:
            preds = self.forward(batch["img"])
        det, items = super().loss(batch, preds)
        logits = self.layout_logits()
        self.model[self.layout_layer].out = None
        n = batch["img"].shape[0]
        target = self.layout_targets(batch, n)
        layout = sum(F.cross_entropy(l.float(), target[:, j]) for j, l in enumerate(logits))
        return torch.cat([det.view(-1), (self.layout_weight * layout * n).view(1)]), items


class LayoutTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = LayoutModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=verbose and RANK == -1)
        if weights:
            model.load(weights)
        kinds, caps = plate_layout_info(Path(self.args.data).parent)
        model.add_layout(kinds, caps)
        print("layout head: caps (digit, district, letter, word) =", caps)
        return model


def layout_probabilities(core, images, imgsz, device):
    """Layout probabilities per image from a trained LayoutModel: list of 4 arrays (one per kind) per image."""
    import cv2
    from ultralytics.data.augment import LetterBox

    box = LetterBox((imgsz, imgsz), auto=False)
    out = []
    core = core.to(device).float().eval()
    with torch.no_grad():
        for start in range(0, len(images), 64):
            batch = []
            for path in images[start:start + 64]:
                img = box(image=cv2.imread(str(path)))[:, :, ::-1].copy()
                batch.append(torch.from_numpy(img).permute(2, 0, 1).float() / 255)
            core(torch.stack(batch).to(device))
            probs = [p.softmax(1).cpu().numpy() for p in core.layout_logits()]
            core.model[core.layout_layer].out = None
            for k in range(len(batch)):
                out.append([p[k] for p in probs])
    return out

In [ ]:
import sys

sys.path.insert(0, str(WORK))
from plate_layers import KINDS as LAYER_KINDS, layout_probabilities

In [ ]:
classes = json.loads((DATA / "classes.json").read_text(encoding="utf-8"))
bangla = [c["bangla"] for c in classes]
kind = [c["kind"] for c in classes]

test_images = sorted((DATA / "images" / "test").iterdir())
print(len(classes), "classes")
print(len(test_images), "test plates")

## Plate text

Same as notebooks 05 and 06: district, word and letter from left to right, then the six digits as `xx-xxxx`.
The true text of a test plate comes from its label file.

In [ ]:
def plate_text(chars):
    # chars: list of (class id, x centre)
    chars = sorted(chars, key=lambda c: c[1])
    head = " ".join(bangla[c] for c, x in chars if kind[c] != "digit")
    digits = "".join(bangla[c] for c, x in chars if kind[c] == "digit")
    if len(digits) == 6:
        digits = digits[:2] + "-" + digits[2:]
    return " ".join(part for part in [head, digits] if part)


def edit_distance(a, b):
    row = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, row[0] = row[0], i
        for j, cb in enumerate(b, 1):
            prev, row[j] = row[j], min(row[j] + 1, row[j - 1] + 1, prev + (ca != cb))
    return row[-1]


def read_labels(image_path, split="test"):
    # YOLO label file -> list of (class id, x centre, y centre, width, height)
    label = DATA / "labels" / split / (image_path.stem + ".txt")
    boxes = []
    for line in label.read_text().splitlines():
        p = line.split()
        if p:
            boxes.append((int(p[0]), *map(float, p[1:])))
    return boxes


def read_truth(image_path, split="test"):
    return plate_text([(b[0], b[1]) for b in read_labels(image_path, split)])


for path in test_images[:3]:
    print(path.name[:50], "->", read_truth(path))
print(edit_distance("ঢাকা মেট্রো গ ২৪-২৯৫৫", "ঢাকা মেট্রো ২৪-২৯৫৫"))

## Baseline: current reader on the test set

The reader at 0.25 confidence, characters sorted left to right, as in the deployed AI service.
This must match notebook 05: 77.4% on all 939 plates (75.7% bd-lpr, 77.9% blpd).

In [ ]:
reader = YOLO(WEIGHTS[0])
device = 0 if torch.cuda.is_available() else "cpu"

rows = []
for start in range(0, len(test_images), 64):
    batch = test_images[start:start + 64]
    results = reader.predict(batch, imgsz=320, conf=0.25, device=device, verbose=False)
    for path, r in zip(batch, results):
        chars = list(zip(r.boxes.cls.int().tolist(), r.boxes.xywhn[:, 0].tolist()))
        truth = read_truth(path)
        pred = plate_text(chars)
        rows.append({"image": path.name, "source": path.name.split("__")[0],
                     "truth": truth, "predicted": pred, "correct": truth == pred,
                     "errors": edit_distance(truth, pred), "length": len(truth)})

baseline = pd.DataFrame(rows)
baseline.head()

In [ ]:
def summary(df):
    return pd.Series({"plates": len(df),
                      "plate acc %": round(100 * df["correct"].mean(), 1),
                      "CER %": round(100 * df["errors"].sum() / df["length"].sum(), 2)})


table = baseline.groupby("source").apply(summary)
table.loc["all"] = summary(baseline)
table

## Plate rules: the 5 best valid readings

A Bangladeshi plate has one district, at most one word (Metro / Chatto / Raj), one series letter and six digits.

The reader is run again at 0.05 confidence so every character position keeps its weaker guesses too.
Guesses that overlap are one position. Each position either takes one of its (up to 3) guesses, scored `log(conf)`,
or is skipped as noise, scored `log(1 - best conf)`. A beam search keeps only readings that follow the plate pattern,
and the 5 best go to the redrawing step.

In [ ]:
def reader_guesses(images, conf=0.05):
    out = {}
    for start in range(0, len(images), 64):
        batch = images[start:start + 64]
        results = reader.predict(batch, imgsz=320, conf=conf, device=device, max_det=100, verbose=False)
        for path, r in zip(batch, results):
            out[path.name] = [
                {"cls": c, "conf": p, "x": xywh[0], "box": xyxy}
                for c, p, xywh, xyxy in zip(r.boxes.cls.int().tolist(), r.boxes.conf.tolist(),
                                            r.boxes.xywhn.tolist(), r.boxes.xyxyn.tolist())
            ]
    return out


test_guesses = reader_guesses(test_images)
print(sum(len(g) for g in test_guesses.values()), "guesses on", len(test_guesses), "plates")

In [ ]:
import math

LIMIT = {"digit": 6, "district": 1, "letter": 1, "word": 1}
KINDS = list(LIMIT)


def iou(a, b):
    w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = w * h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union else 0.0


def positions(guesses):
    # overlapping guesses are one character position, best guess first
    groups = []
    for g in sorted(guesses, key=lambda g: -g["conf"]):
        for group in groups:
            if iou(group[0]["box"], g["box"]) > 0.5:
                group.append(g)
                break
        else:
            groups.append([g])
    return sorted(groups, key=lambda group: group[0]["x"])


def top_readings(guesses, k=5, beam=256):
    groups = positions(guesses)
    # a state is (score, counts per kind, chosen guesses)
    states = [(0.0, (0, 0, 0, 0), ())]
    for i, group in enumerate(groups):
        left = len(groups) - i - 1
        options = group[:3]
        new_states = []
        for score, counts, chosen in states:
            # skip this position
            new_states.append((score + math.log(max(1e-4, 1 - options[0]["conf"])), counts, chosen))
            # or take one of its guesses
            for g in options:
                j = KINDS.index(kind[g["cls"]])
                c = list(counts)
                c[j] += 1
                if c[j] <= LIMIT[KINDS[j]]:
                    new_states.append((score + math.log(max(1e-4, g["conf"])), tuple(c), chosen + (g,)))
        # drop states that can no longer reach six digits, a district and a letter
        new_states = [s for s in new_states if (6 - s[1][0]) + (1 - s[1][1]) + (1 - s[1][2]) <= left]
        states = sorted(new_states, key=lambda s: -s[0])[:beam]

    readings = {}
    for score, counts, chosen in states:
        if counts[:3] == (6, 1, 1):
            text = plate_text([(g["cls"], g["x"]) for g in chosen])
            if text not in readings or score > readings[text]["logp"]:
                readings[text] = {"text": text, "logp": score, "chars": [(g["cls"], g["box"]) for g in chosen]}
    return sorted(readings.values(), key=lambda r: -r["logp"])[:k]


example = test_images[0].name
for r in top_readings(test_guesses[example]):
    print(round(r["logp"], 2), r["text"])
print("truth:", read_truth(test_images[0]))

In [ ]:
base_pred = dict(zip(baseline["image"], baseline["predicted"]))

rules = []
for path in test_images:
    readings = top_readings(test_guesses[path.name])
    truth = read_truth(path)
    # no valid reading: keep what the reader says today
    pred = readings[0]["text"] if readings else base_pred[path.name]
    rules.append({"image": path.name, "source": path.name.split("__")[0], "truth": truth,
                  "predicted": pred, "correct": truth == pred,
                  "errors": edit_distance(truth, pred), "length": len(truth),
                  "in_top5": any(r["text"] == truth for r in readings) or truth == pred,
                  "n_readings": len(readings)})
rules = pd.DataFrame(rules)

compare = pd.DataFrame({
    "baseline acc %": baseline.groupby("source")["correct"].mean() * 100,
    "plate rules acc %": rules.groupby("source")["correct"].mean() * 100,
    "truth in top 5 %": rules.groupby("source")["in_top5"].mean() * 100,
})
compare.loc["all"] = [baseline["correct"].mean() * 100, rules["correct"].mean() * 100, rules["in_top5"].mean() * 100]
print("plates with no valid reading (reader text kept):", (rules["n_readings"] == 0).sum())
print("CER %: baseline", round(100 * baseline["errors"].sum() / baseline["length"].sum(), 2),
      "| plate rules", round(100 * rules["errors"].sum() / rules["length"].sum(), 2))
compare.round(1)

## Where the wrong plates are lost

Every plate the plate rules still get wrong is put in one group, checked in this order:

- **not the plate pattern**: the true label itself does not have one district, one letter and six digits
- **character not found**: a true character has no reader guess on it at all (even at 0.05)
- **right class not guessed**: the reader guessed something there, but never the right class
- **right class ranked lower**: the right class is among the guesses at that place, but a wrong one was chosen
- **other**: every character has the right class as its best guess, yet the text is wrong (an extra character, order)

In [ ]:
def corners(box):
    x, y, w, h = box
    return [x - w / 2, y - h / 2, x + w / 2, y + h / 2]


def why_wrong(path):
    labels = read_labels(path)
    counts = {k: sum(kind[b[0]] == k for b in labels) for k in LIMIT}
    if (counts["digit"], counts["district"], counts["letter"]) != (6, 1, 1) or counts["word"] > 1:
        return "not the plate pattern"
    guesses = test_guesses[path.name]
    worst = "other"
    for cls, *box in labels:
        here = sorted([g for g in guesses if iou(g["box"], corners(box)) > 0.3], key=lambda g: -g["conf"])
        if not here:
            return "character not found"
        if cls not in [g["cls"] for g in here]:
            worst = "right class not guessed"
        elif here[0]["cls"] != cls and worst == "other":
            worst = "right class ranked lower"
    return worst


wrong = rules[~rules["correct"]].copy()
wrong["why"] = [why_wrong(DATA / "images" / "test" / name) for name in wrong["image"]]
print(len(wrong), "wrong plates out of", len(rules))
wrong.groupby(["why", "source"]).size().unstack(fill_value=0).assign(total=lambda d: d.sum(axis=1))

In [ ]:
for why, group in wrong.groupby("why"):
    print(why)
    print(group[["truth", "predicted"]].head(5).to_string(index=False))
    print()

## Learned plate layout

The fixed rules above force exactly six digits, one district and one letter, so plates with another layout
(for example `ব ৫৭৩৮`, four digits and no district) can never be read. Here the layout is **learned from the training
labels** instead: every training plate is counted by its number of digits, districts, letters and words.

The search then allows any count up to the largest seen in training, and each reading gets
`score = reader log-probability + λ × log P(layout)`. Common layouts are preferred, rare ones are still possible,
and an unseen layout gets a small smoothed probability. `λ` is chosen on validation.

In [ ]:
from collections import Counter

layout_counts = Counter()
for path in sorted((DATA / "images" / "train").iterdir()):
    labels = read_labels(path, "train")
    layout_counts[tuple(sum(kind[b[0]] == k for b in labels) for k in KINDS)] += 1

n_train = sum(layout_counts.values())
CAP = {k: max(layout[j] for layout in layout_counts) for j, k in enumerate(KINDS)}
n_possible = math.prod(CAP[k] + 1 for k in KINDS)


def layout_logp(counts):
    # add-0.5 smoothing over every layout within the caps
    return math.log((layout_counts.get(counts, 0) + 0.5) / (n_train + 0.5 * n_possible))


print(n_train, "training plates,", len(layout_counts), "different layouts, caps", CAP)
pd.DataFrame([{"digits": l[0], "districts": l[1], "letters": l[2], "words": l[3], "plates": n,
               "share %": round(100 * n / n_train, 2)} for l, n in layout_counts.most_common(12)])

In [ ]:
import numpy as np


def top_readings_learned(guesses, lam, k=5, beam=256, prior=None):
    prior = prior or layout_logp
    groups = positions(guesses)
    states = [(0.0, (0, 0, 0, 0), ())]   # (score, counts per kind, chosen guesses)
    for group in groups:
        options = group[:3]
        new_states = []
        for score, counts, chosen in states:
            new_states.append((score + math.log(max(1e-4, 1 - options[0]["conf"])), counts, chosen))   # skip
            for g in options:
                j = KINDS.index(kind[g["cls"]])
                c = list(counts)
                c[j] += 1
                if c[j] <= CAP[KINDS[j]]:
                    new_states.append((score + math.log(max(1e-4, g["conf"])), tuple(c), chosen + (g,)))
        states = sorted(new_states, key=lambda s: -s[0])[:beam]

    readings = {}
    for score, counts, chosen in states:
        if not chosen:
            continue
        total = score + lam * prior(counts)
        text = plate_text([(g["cls"], g["x"]) for g in chosen])
        if text not in readings or total > readings[text]["logp"]:
            readings[text] = {"text": text, "logp": total, "layout": counts,
                              "chars": [(g["cls"], g["box"]) for g in chosen]}
    return sorted(readings.values(), key=lambda r: -r["logp"])[:k]


val_images = sorted((DATA / "images" / "val").iterdir())
val_guesses = reader_guesses(val_images)
val_truth = {p.name: read_truth(p, "val") for p in val_images}


def first_text(readings):
    return readings[0]["text"] if readings else ""


LAMBDAS = [0.25, 0.5, 1, 1.5, 2, 3, 5, 8]
val_acc = {lam: np.mean([first_text(top_readings_learned(val_guesses[n], lam)) == t for n, t in val_truth.items()])
           for lam in LAMBDAS}
best_lam = max(val_acc, key=val_acc.get)
for lam, a in val_acc.items():
    print(f"lambda {lam:>4}: validation plate accuracy {100 * a:.2f}%", "<- best" if lam == best_lam else "")

In [ ]:
fixed_pred = dict(zip(rules["image"], rules["predicted"]))

learned = []
for path in test_images:
    readings = top_readings_learned(test_guesses[path.name], best_lam)
    truth = read_truth(path)
    pred = first_text(readings) or base_pred[path.name]
    learned.append({"image": path.name, "source": path.name.split("__")[0], "truth": truth, "predicted": pred,
                    "correct": truth == pred, "errors": edit_distance(truth, pred), "length": len(truth),
                    "in_top5": any(r["text"] == truth for r in readings) or truth == pred})
learned = pd.DataFrame(learned)


def acc_cer(df):
    return (round(100 * df["correct"].mean(), 1), round(100 * df["errors"].sum() / df["length"].sum(), 2))


rows = []
for name, df in [("baseline (today)", baseline), ("fixed plate rules", rules), ("learned layout", learned)]:
    for src in ["all"] + sorted(df["source"].unique()):
        part = df if src == "all" else df[df["source"] == src]
        a, c = acc_cer(part)
        rows.append({"method": name, "plates": src, "acc %": a, "CER %": c})
rows.append({"method": "learned layout: truth in top 5", "plates": "all",
             "acc %": round(100 * learned["in_top5"].mean(), 1), "CER %": None})
print(f"lambda = {best_lam}")
pd.DataFrame(rows).pivot(index="method", columns="plates", values="acc %")

In [ ]:
# plates the fixed rules could not read because their layout is not 6 digits / 1 district / 1 letter
other_layout = wrong[wrong["why"] == "not the plate pattern"]["image"]
check = learned.set_index("image").loc[other_layout, ["truth", "predicted", "correct"]]
print(f"{check['correct'].sum()} of {len(check)} 'not the plate pattern' plates are now read correctly")
print(check.head(10).to_string(index=False))

# from here on the redraw step works on the learned readings
fixed_top_readings = top_readings
top_readings = lambda guesses, k=5: top_readings_learned(guesses, best_lam, k)


def readings_for(name, guesses):
    return top_readings_learned(guesses, best_lam)

## Learned layout from the layout head

Only for a reader trained with the layout head (notebook 13). Instead of one table of layout counts for all plates,
the head predicts the layout of **each plate**, and a reading is scored by
`reader log-probability + λ × log P(layout | this plate)`. `λ` is chosen on validation. If the head is better on
validation than the counted layout, the redraw step uses it.

In [ ]:
core = YOLO(WEIGHTS[0]).model
HAS_HEAD = hasattr(core, "layout_head")
print("reader has a layout head:", HAS_HEAD)

if HAS_HEAD:
    head_images = val_images + test_images
    head_probs = dict(zip([p.name for p in head_images],
                          layout_probabilities(core, head_images, 320, device_t if "device_t" in globals() else
                                               ("cuda:0" if torch.cuda.is_available() else "cpu"))))

    def head_prior(name):
        p = head_probs[name]
        def prior(counts):
            return sum(math.log(max(1e-6, float(p[j][c]))) if c < len(p[j]) else math.log(1e-6)
                       for j, c in enumerate(counts))
        return prior

    head_acc = {lam: np.mean([first_text(top_readings_learned(val_guesses[n], lam, prior=head_prior(n))) == t
                              for n, t in val_truth.items()]) for lam in LAMBDAS}
    head_lam = max(head_acc, key=head_acc.get)
    for lam, a in head_acc.items():
        print(f"lambda {lam:>4}: validation plate accuracy {100 * a:.2f}%", "<- best" if lam == head_lam else "")

    head_rows = []
    for path in test_images:
        readings = top_readings_learned(test_guesses[path.name], head_lam, prior=head_prior(path.name))
        truth = read_truth(path)
        pred = first_text(readings) or base_pred[path.name]
        head_rows.append({"image": path.name, "source": path.name.split("__")[0], "truth": truth, "predicted": pred,
                          "correct": truth == pred, "errors": edit_distance(truth, pred), "length": len(truth)})
    head_layout = pd.DataFrame(head_rows)

    rows = []
    for name, df in [("baseline (today)", baseline), ("fixed plate rules", rules),
                     ("learned layout (counts)", learned), ("learned layout (head)", head_layout)]:
        for src in ["all"] + sorted(df["source"].unique()):
            part = df if src == "all" else df[df["source"] == src]
            a, c = acc_cer(part)
            rows.append({"method": name, "plates": src, "acc %": a, "CER %": c})
    display(pd.DataFrame(rows).pivot(index="method", columns="plates", values="acc %"))

    if max(head_acc.values()) > val_acc[best_lam]:
        print("the head is better on validation: the redraw step uses it")

        def readings_for(name, guesses):
            return top_readings_learned(guesses, head_lam, prior=head_prior(name))

## Glyph atlas

To draw a plate we need one clean image per character class. For each class we cut its boxes out of the training crops,
keep the 40 tallest (the sharpest), and take their pixel-wise median. No network is trained here.

Every crop is first turned grey, its contrast stretched to 0–1, and flipped if needed so the ink is dark on a light plate.

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

PH, PW = 80, 160   # every plate is compared at this size
GLYPH = 48         # size of one atlas image
PAD = 0.08         # margin around a character box, the same when cutting and drawing


def stretch(gray):
    lo, hi = np.percentile(gray, [2, 98])
    return np.clip((gray - lo) / max(hi - lo, 1e-3), 0, 1).astype(np.float32)


def normalise(gray):
    # contrast 0-1, ink dark on a light plate. The light/dark decision uses only the middle of the crop,
    # because many crops have a thick black frame around the plate.
    g = stretch(gray)
    h, w = g.shape
    middle = g[int(0.2 * h):int(0.8 * h), int(0.15 * w):int(0.85 * w)]
    if np.median(middle) < 0.5:
        g = 1 - g
    return g


def load_plate(path):
    # returns the normalised crop at full size and at the compare size
    full = normalise(np.asarray(Image.open(path).convert("L"), np.float32) / 255)
    small = Image.fromarray((full * 255).astype(np.uint8)).resize((PW, PH), Image.BILINEAR)
    return full, np.asarray(small, np.float32) / 255


def to_corners(box):
    # YOLO (x centre, y centre, w, h) -> (x0, y0, x1, y1), all relative to the crop
    x, y, w, h = box
    return (x - w / 2, y - h / 2, x + w / 2, y + h / 2)


def padded(box):
    x0, y0, x1, y1 = box
    dx, dy = (x1 - x0) * PAD, (y1 - y0) * PAD
    return (x0 - dx, y0 - dy, x1 + dx, y1 + dy)


def resize(img, w, h):
    return np.asarray(Image.fromarray((img * 255).astype(np.uint8)).resize((w, h), Image.BILINEAR), np.float32) / 255


def clean_glyph(patch):
    # one character: own contrast, ink dark (ink is the smaller part of a character box), ink centred
    g = stretch(patch)
    if g.mean() < 0.5:
        g = 1 - g
    g = resize(g, GLYPH, GLYPH)
    ink = 1 - g
    if ink.sum() > 0:
        ys, xs = np.indices(ink.shape)
        dy = int(round(GLYPH / 2 - (ys * ink).sum() / ink.sum()))
        dx = int(round(GLYPH / 2 - (xs * ink).sum() / ink.sum()))
        shifted = np.ones_like(g)
        src = g[max(0, -dy):GLYPH - max(0, dy), max(0, -dx):GLYPH - max(0, dx)]
        shifted[max(0, dy):max(0, dy) + src.shape[0], max(0, dx):max(0, dx) + src.shape[1]] = src
        g = shifted
    return g


def sharpness(g):
    # variance of the Laplacian: higher is sharper
    lap = g[1:-1, 1:-1] * 4 - g[:-2, 1:-1] - g[2:, 1:-1] - g[1:-1, :-2] - g[1:-1, 2:]
    return float(lap.var())

In [ ]:
from collections import defaultdict

train_images = sorted((DATA / "images" / "train").iterdir())
pool = defaultdict(list)   # class -> [(sharpness, glyph image)]

for path in train_images:
    gray = np.asarray(Image.open(path).convert("L"), np.float32) / 255
    h, w = gray.shape
    for cls, *box in read_labels(path, "train"):
        if len(pool[cls]) >= 400:
            continue
        x0, y0, x1, y1 = padded(to_corners(box))
        x0, x1 = int(max(0, x0 * w)), int(min(w, x1 * w))
        y0, y1 = int(max(0, y0 * h)), int(min(h, y1 * h))
        if x1 - x0 >= 8 and y1 - y0 >= 12:   # skip tiny characters
            g = clean_glyph(gray[y0:y1, x0:x1])
            pool[cls].append((sharpness(g), g))

atlas = {}
for cls, items in pool.items():
    items.sort(key=lambda t: -t[0])
    median = np.median(np.stack([g for _, g in items[:40]]), axis=0)
    # final stretch so the background is pure white and the ink black
    lo, hi = np.percentile(median, [3, 85])
    atlas[cls] = np.clip((median - lo) / max(hi - lo, 1e-3), 0, 1).astype(np.float32)

missing = [classes[c]["name"] for c in range(len(classes)) if c not in atlas]
print(len(train_images), "training crops")
print(len(atlas), "of", len(classes), "classes in the atlas, missing:", missing or "none")
np.savez_compressed(WORK / "glyph_atlas.npz", **{str(c): g for c, g in atlas.items()})

In [ ]:
cols = 16
rows = math.ceil(len(atlas) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 0.9, rows))
for ax in axes.flat:
    ax.axis("off")
for ax, cls in zip(axes.flat, sorted(atlas)):
    ax.imshow(atlas[cls], cmap="gray", vmin=0, vmax=1)
    ax.set_title(classes[cls]["name"][:8], fontsize=6)
plt.tight_layout()
plt.savefig(WORK / "atlas.png", dpi=130)
plt.show()